# 🛡️ CyberSec AI 10-Weeks Interactive Colab Notebook
## An Ninh Mạng & Ứng Dụng AI Thế Hệ Mới (Dành Cho Mobile & Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Notebook này cho phép học viên thực thi trực tiếp toàn bộ các bài thực hành Python từ Tuần 1 đến Tuần 10 (Socket, Scapy, Nmap Audit, Bcrypt Hashing, OSINT Prompting, Web Log Audit, và AI Anomaly Detection) trên môi trường điện toán đám mây Google Colab từ máy tính hoặc điện thoại.

### 📦 Bước 1: Cài đặt công cụ Nmap & các thư viện Python

In [ ]:
# Cài đặt nmap và các thư viện cần thiết trong Colab
!apt-get update -qq && !apt-get install -y -qq nmap > /dev/null
!pip install -q scapy bcrypt pandas scikit-learn requests matplotlib

### 🔌 Tuần 1: Lập Trình Python Socket Communication & TCP Echo Server Lab

#### Cell 1 - TCP Echo Server (chạy nền trên 127.0.0.1:9999)

In [ ]:
import socket
import threading

HOST = "127.0.0.1"
PORT = 9999

def handle_client(conn, addr):
    print(f"[SERVER] Connected: {addr}")
    while True:
        data = conn.recv(1024)
        if not data:
            break
        print(f"[SERVER] Received: {data}")
        conn.sendall(data)
    conn.close()
    print(f"[SERVER] Closed: {addr}")

def start_server():
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((HOST, PORT))
    server.listen()
    print(f"[SERVER] Listening on {HOST}:{PORT}")
    while True:
        conn, addr = server.accept()
        t = threading.Thread(target=handle_client, args=(conn, addr), daemon=True)
        t.start()

threading.Thread(target=start_server, daemon=True).start()

#### Cell 2 - TCP Client gửi 5 gói tin

In [ ]:
import socket
import time

TEST_PAYLOADS = [
    b"HELLO",
    b"TCP LAB",
    bytes([0, 1, 2, 3, 255]),
    b"CyberSecurity Week02",
    bytes(range(32))
]

rtts = []
for i, payload in enumerate(TEST_PAYLOADS, start=1):
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    start = time.perf_counter()
    client.connect(("127.0.0.1", 9999))
    client.sendall(payload)
    response = client.recv(1024)
    end = time.perf_counter()
    rtt_ms = (end - start) * 1000
    rtts.append(rtt_ms)
    print(f"\nPacket #{i}")
    print(f"Sent     : {payload}")
    print(f"Received : {response}")
    print(f"RTT      : {rtt_ms:.3f} ms")
    client.close()

#### Cell 3 - Phân tích độ trễ RTT (Min/Max/Avg)

In [ ]:
print("\n===== RTT SUMMARY =====")
print(f"Min RTT : {min(rtts):.3f} ms")
print(f"Max RTT : {max(rtts):.3f} ms")
print(f"Avg RTT : {sum(rtts)/len(rtts):.3f} ms")

#### Cell 4 - Vẽ biểu đồ RTT bằng Matplotlib

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(range(1, len(rtts)+1), rtts, marker="o")
plt.xlabel("Packet Number")
plt.ylabel("RTT (ms)")
plt.title("TCP Echo Server Round-Trip Time")
plt.xticks(range(1, 6))
plt.grid(True)
plt.show()

### 🔎 Tuần 2: Fast Port Scanner đa luồng

In [ ]:
import concurrent.futures

COMMON_PORTS = [21, 22, 80, 443, 8080]

def scan_port(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.5)
    res = s.connect_ex(('127.0.0.1', port))
    s.close()
    return port, (res == 0)

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(scan_port, COMMON_PORTS)
    for p, open_status in results:
        print(f"[+] Port {p:5d} : {'OPEN' if open_status else 'CLOSED'}")

### 🔍 Tuần 5: Quét Nmap Tự Động Trên Localhost

In [ ]:
import subprocess

res = subprocess.run(["nmap", "-sV", "--open", "-p", "1-100", "127.0.0.1"], capture_output=True, text=True)
print(res.stdout)

### 🌐 Tuần 6: Phân Tích Gói Tin Scapy

In [ ]:
from scapy.all import IP, TCP, Ether

# Sinh gói tin giả lập để kiểm tra cấu trúc
pkt = Ether()/IP(dst="127.0.0.1")/TCP(dport=80, flags="S")
print(f"[+] Simulated Packet Summary: {pkt.summary()}")
print(f"[+] IP Source: {pkt[IP].src} -> IP Dst: {pkt[IP].dst}")

### 🔐 Tuần 7: Mã Hóa Bcrypt Mật Khẩu

In [ ]:
import bcrypt

pw = "UserSecretPassword@2026"
salt = bcrypt.gensalt(rounds=10)
hashed = bcrypt.hashpw(pw.encode('utf-8'), salt)

print(f"[+] Bcrypt Salted Hash: {hashed.decode('utf-8')}")
print(f"[+] Verify Result     : {bcrypt.checkpw(pw.encode('utf-8'), hashed)}")

### 🤖 Tuần 8: Tự Động Trích Xuất OSINT Threat Intel

In [ ]:
import re, json

sample_report = "Threat IP 192.168.1.100 connected to C2 45.33.32.156 with SHA256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855."
ips = re.findall(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b', sample_report)
hashes = re.findall(r'\b[A-Fa-f0-9]{64}\b', sample_report)

print(json.dumps({"extracted_ips": ips, "extracted_hashes": hashes}, indent=2))

### 📝 Tuần 9: Parse Web Access Log Phát Hiện Tấn Công

In [ ]:
logs = [
    '127.0.0.1 - - [27/Jul/2026] "GET /index.html HTTP/1.1" 200 1024',
    '127.0.0.1 - - [27/Jul/2026] "GET /login?user=admin\' OR 1=1-- HTTP/1.1" 200 512'
]

sqli_regex = re.compile(r"(?i)(\'|OR%201=1|UNION|SELECT)")
for l in logs:
    if sqli_regex.search(l):
        print(f"[⚠️ ALERT - SQL INJECTION]: {l}")
    else:
        print(f"[✅ BENIGN]: {l}")

### 🤖 Tuần 10: AI SOC Monitoring (Isolation Forest Anomaly Detection)

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest

# Synthetic data generation
normal_reqs = np.random.normal(75, 10, 95)
anomaly_reqs = np.random.normal(500, 50, 5)
reqs = np.concatenate([normal_reqs, anomaly_reqs])

df = pd.DataFrame({'requests_per_min': reqs})
model = IsolationForest(contamination=0.05, random_state=42)
df['anomaly'] = model.fit_predict(df[['requests_per_min']])

anomalies = df[df['anomaly'] == -1]
print(f"[+] Total records: {len(df)}")
print(f"[⚠️ ALERT] Anomalies Detected by Isolation Forest: {len(anomalies)}")
print(anomalies.head())